# Colab preflight (optional diagnostic)

Use this only to diagnose GPU, Drive, clone, or package setup. It is **not** a prerequisite for the canonical reproduction notebook, which includes its own setup: [`01_reproduce_mft_gemma3.ipynb`](https://colab.research.google.com/github/rlogger/em-displacement-vlm/blob/main/notebooks/01_reproduce_mft_gemma3.ipynb).

Runtime → GPU → **A100** recommended.

In [ ]:
!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 1. Drive mount

In [ ]:
from pathlib import Path
import os

MOUNT_DRIVE = True
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    for sub in ("data", "checkpoints", "results"):
        (DRIVE_PROJECT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
    os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
    os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")
    print("Drive project:", DRIVE_PROJECT)
else:
    print("Drive mount skipped — using /content for ephemeral storage.")

## 2. Resolve a clean, immutable repository commit

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/rlogger/em-displacement-vlm.git"
REPO_DIR = Path("/content/em-displacement-vlm")
REPO_REF = "main"  # Or an exact 40-character commit.

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise SystemExit(f"{REPO_DIR} exists but is not a git clone; restart Colab.")
    origin = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    canonical_origins = {REPO_URL.rstrip("/"), REPO_URL.removesuffix(".git").rstrip("/")}
    if origin.rstrip("/") not in canonical_origins:
        raise SystemExit(f"Unexpected origin {origin!r}; restart with the canonical clone.")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain=v1", "--untracked-files=all"],
        text=True,
    ).strip()
    if dirty:
        raise SystemExit("Existing clone is dirty; restart Colab instead of repairing it in place.")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--prune", "--tags", "origin"])
else:
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)])

target = "origin/main" if REPO_REF == "main" else REPO_REF
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", f"{target}^{{commit}}"], text=True
).strip()
subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "--detach", commit])
%cd {REPO_DIR}

print("Resolved commit:", commit)
subprocess.check_call(["git", "status", "--short"] )

## 3. Install

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

%pip install -q -e ".[vlm,dev]"

from em_displacement_vlm.runtime import runtime_info
from em_displacement_vlm.paths import data_dir, checkpoint_dir

for k, v in runtime_info().items():
    print(f"{k}: {v}")
print("data_dir:", data_dir())
print("checkpoint_dir:", checkpoint_dir())
print("\nFor the canonical M_ft reproduction open: notebooks/01_reproduce_mft_gemma3.ipynb")

## 4. Secrets

In [ ]:
from google.colab import userdata
import os

def _set_secret(name: str) -> None:
    try:
        os.environ[name] = userdata.get(name)
        print(f"Loaded secret: {name}")
    except Exception:
        print(f"Secret not set (ok if unused): {name}")

for key in ("HF_TOKEN",):
    _set_secret(key)

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face session ready without writing Git credentials.")